# Your first notebook on HPE Private Cloud AI

This notebook checks that your notebook server can reach a model served by **HPE MLIS**, then runs a small **LangGraph** agent on it.

**You will need** (ask your trainer or administrator if you do not have them):

1. The **endpoint URL** of the MLIS deployment
2. The **model name** served by that deployment
3. A **deployment token** for it
4. Optional: the endpoint, model name and token of an **embedding** deployment

**How to use it:** run one cell at a time with `Shift + Enter`. Read the "Expected output" note under each step before you move on.

Nothing in this notebook stores your token. Do not paste it into a cell.

## Step 1. Check where you are running

This cell prints facts about the machine your kernel runs on. Your code runs inside the platform, not on your laptop.

**Expected output:** a Python version of 3.10 or later, a hostname that looks like a notebook pod name, and the path of your working folder.

In [ ]:
import os
import platform
import shutil
import sys

print("Python        :", sys.version.split()[0])
print("Platform      :", platform.platform())
print("Hostname      :", platform.node())
print("Working folder:", os.getcwd())
print("nvidia-smi    :", "found" if shutil.which("nvidia-smi") else "not found (normal: the model runs on MLIS, not in this notebook)")

## Step 2. Install the packages

`%pip install` installs into the notebook's base environment. Packages installed this way are **removed when the notebook server restarts**, so this cell must be re-run after a restart. Installing needs access to a package index; it does not work in air-gapped environments.

**Expected output:** a short pause and no red error text. If a later cell says `ModuleNotFoundError`, restart the kernel (Kernel menu, Restart Kernel) and continue from Step 3.

In [ ]:
%pip install --quiet langchain-openai langgraph requests

## Step 3. Enter your endpoint details

The cell reads each setting from an environment variable if one exists (an empty value means "skip"), and otherwise asks you. The token prompt hides what you type.

Where to find the values in HPE AI Essentials:

- **Endpoint URL and model name:** MLIS, Deployments, open your deployment
- **Deployment token:** MLIS, create a deployment token for that deployment, and copy it when it is shown

Paste the endpoint as shown. If it ends in `/chat/completions`, the cell trims that part.

**Expected output:** a summary of what was entered, with the token hidden.

In [ ]:
import getpass
import os


def ask(name, prompt, secret=False, optional=False):
    """Read a setting from an environment variable, or ask for it."""
    if name in os.environ:
        value = os.environ[name].strip()
    else:
        value = (getpass.getpass(prompt) if secret else input(prompt)).strip()
    if not value and not optional:
        raise ValueError(f"{name} is required")
    return value


def clean_base_url(url):
    """Remove a trailing slash and any /chat/completions or /embeddings suffix."""
    url = url.strip().rstrip("/")
    for suffix in ("/chat/completions", "/embeddings"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


LLM_BASE_URL = clean_base_url(ask("MLIS_LLM_BASE_URL", "LLM endpoint URL: "))
LLM_MODEL = ask("MLIS_LLM_MODEL", "LLM model name: ")
TOKEN = ask("MLIS_DEPLOY_TOKEN", "MLIS deployment token (hidden): ", secret=True)

emb_url = ask("MLIS_EMB_BASE_URL", "Embedding endpoint URL (press Enter to skip): ", optional=True)
EMB_BASE_URL = clean_base_url(emb_url) if emb_url else ""
EMB_MODEL = ask("MLIS_EMB_MODEL", "Embedding model name: ", optional=True) if EMB_BASE_URL else ""
EMB_TOKEN = (ask("MLIS_EMB_TOKEN", "Embedding token (Enter to reuse the LLM token): ", secret=True, optional=True) or TOKEN) if EMB_BASE_URL else ""

print("LLM endpoint      :", LLM_BASE_URL)
print("LLM model         :", LLM_MODEL)
print("Token entered     :", "yes" if TOKEN else "no", f"({len(TOKEN)} characters)")
print("Embedding endpoint:", EMB_BASE_URL or "skipped")

## Step 4. Make one raw call to the model

This uses plain `requests`, so you can see exactly what goes over the network: a `POST` to `<endpoint>/chat/completions` with the token in an `Authorization: Bearer` header.

Some models (for example Qwen3) write a reasoning block inside `<think>` tags before the answer. The helper `strip_reasoning` removes it so that later steps see only the answer.

**Expected output:** the word `ready` and a token-usage dictionary. If you get an error, read the **Hint** line in the message first.

In [ ]:
import re
import requests

HINTS = {
    401: "The token is missing or wrong. Copy the deployment token again.",
    403: "The token is not allowed for this deployment, or it has expired.",
    404: "Wrong URL path or model name. Try adding or removing /v1 at the end of the endpoint.",
    429: "Too many requests. Other participants share this endpoint. Wait a few seconds and retry.",
    502: "Gateway error. The deployment may still be starting.",
    503: "The deployment is not ready (starting, scaled to zero, or overloaded). Check its status in MLIS.",
}


def strip_reasoning(text):
    """Remove a leading <think>...</think> block that some models emit."""
    return re.sub(r"<think>.*?(?:</think>|$)", "", text or "", flags=re.DOTALL).strip()


def post_json(url, token, payload, timeout=90):
    resp = requests.post(
        url,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=payload,
        timeout=timeout,
    )
    if not resp.ok:
        hint = HINTS.get(resp.status_code, "See the response body below.")
        raise RuntimeError(f"HTTP {resp.status_code} from {url}\nHint: {hint}\nBody: {resp.text[:500]}")
    return resp.json()


reply = post_json(
    f"{LLM_BASE_URL}/chat/completions",
    TOKEN,
    {
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
        "temperature": 0,
        "max_tokens": 256,
    },
)
print("Answer:", strip_reasoning(reply["choices"][0]["message"]["content"]))
print("Usage :", reply.get("usage"))

## Step 5. Make the same call through LangChain

Agent frameworks talk to the same endpoint. `ChatOpenAI` accepts a custom `base_url`, which is how you point LangChain and LangGraph at a model served on MLIS.

**Expected output:** a one-sentence answer and a usage summary.

**If this step fails but Step 4 worked:** the most common cause is a certificate that the Python HTTP client does not trust (private clouds often use an internal certificate authority). Ask your administrator for the CA file, then run `import os; os.environ['SSL_CERT_FILE'] = os.environ['REQUESTS_CA_BUNDLE'] = '/path/to/ca.pem'` in a cell before Step 4 and re-run from there.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url=LLM_BASE_URL,
    api_key=TOKEN,          # sent as: Authorization: Bearer <token>
    model=LLM_MODEL,
    temperature=0,
    max_tokens=512,
    timeout=90,
    max_retries=1,
)

answer = llm.invoke("In one sentence, what is an AI agent?")
print(strip_reasoning(answer.content))
print("Usage:", answer.usage_metadata)

## Step 6. Call the embedding model (optional)

Retrieval features such as long-term memory turn text into vectors. This step calls an embedding deployment through the same kind of request. NVIDIA retrieval embedding models usually expect an `input_type` field (`query` or `passage`); the helper retries without it if the server rejects it.

**Expected output:** the vector length (for example 1024) and the first five values. If you skipped the embedding endpoint in Step 3, the cell says so and you can move on.

In [ ]:
def embed(texts, input_type="query"):
    payload = {"model": EMB_MODEL, "input": texts, "encoding_format": "float"}
    url = f"{EMB_BASE_URL}/embeddings"
    try:
        return post_json(url, EMB_TOKEN, {**payload, "input_type": input_type})
    except RuntimeError as err:
        if "HTTP 400" in str(err) or "HTTP 422" in str(err):
            return post_json(url, EMB_TOKEN, payload)
        raise


if EMB_BASE_URL:
    out = embed(["reset a user password"])
    vector = out["data"][0]["embedding"]
    print("Vector length:", len(vector))
    print("First 5 values:", [round(v, 4) for v in vector[:5]])
else:
    print("Skipped: no embedding endpoint was configured in Step 3.")

## Step 7. Run a small LangGraph agent

This is a two-path graph for the IT incident use case:

```
START -> classify -> (high severity)  -> escalate -> END
                  -> (other severity) -> resolve  -> END
```

- `classify` asks the model for a severity: low, medium or high.
- `route` decides the next node from the state. Routing on a plain value in the state keeps the decision deterministic.
- `escalate` stops and asks for a human. No system change is made.
- `resolve` asks the model for a first-line fix.

**Expected output:** a Mermaid diagram of the graph as text, then one result per incident. A small model can misjudge severity, so treat the classification as a starting point for discussion.

In [ ]:
import re
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class IncidentState(TypedDict, total=False):
    incident: str
    severity: str
    action: str


def classify(state: IncidentState) -> dict:
    prompt = (
        "Classify the severity of this IT incident as exactly one word: low, medium or high.\n"
        f"Incident: {state['incident']}"
    )
    text = strip_reasoning(llm.invoke(prompt).content).lower()
    match = re.search(r"\b(high|medium|low)\b", text)
    return {"severity": match.group(1) if match else "medium"}


def route(state: IncidentState) -> Literal["escalate", "resolve"]:
    return "escalate" if state["severity"] == "high" else "resolve"


def resolve(state: IncidentState) -> dict:
    prompt = f"Give a two-sentence first-line fix for this IT incident: {state['incident']}"
    return {"action": strip_reasoning(llm.invoke(prompt).content)}


def escalate(state: IncidentState) -> dict:
    return {"action": "ESCALATED: waiting for human approval before any system change."}


graph = StateGraph(IncidentState)
graph.add_node("classify", classify)
graph.add_node("resolve", resolve)
graph.add_node("escalate", escalate)
graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {"escalate": "escalate", "resolve": "resolve"})
graph.add_edge("resolve", END)
graph.add_edge("escalate", END)
app = graph.compile()

print(app.get_graph().draw_mermaid())

In [ ]:
incidents = [
    "The payments API is down for all customers and the error rate is 100 percent.",
    "A user in the finance office cannot see the new shared printer.",
]

results = []
for text in incidents:
    final = app.invoke({"incident": text})
    results.append(final)
    print("Incident:", final["incident"])
    print("Severity:", final["severity"])
    print("Action  :", final["action"])
    print("-" * 60)

## Step 8. Save the results

Files on your private notebook volume are still there after the server is stopped and started again. Ask your trainer which folder that is; if your working folder is not on it, save the file there instead. The token is never written.

**Expected output:** the full path of the saved JSON file. You can see it in the JupyterLab file browser on the left.

In [ ]:
import datetime
import json
import os

path = f"pcai_first_run_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"
with open(path, "w") as handle:
    json.dump({"model": LLM_MODEL, "results": results}, handle, indent=2)

print("Saved:", os.path.abspath(path))

## Step 9. Finish

1. Save the notebook (File menu, Save Notebook).
2. If your trainer asks for it, use Kernel, Restart Kernel and Run All Cells to prove it runs from a clean start.
3. When you are done for the day, return to the **Notebook Servers** screen in HPE AI Essentials and **stop** your server to release CPU and memory. Your files stay in your volume.

### Quick troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| `HTTP 401` or `403` | Wrong or expired token | Create or copy the deployment token again |
| `HTTP 404` | Wrong URL path or model name | Add or remove `/v1`; copy the model name exactly |
| `HTTP 503`, timeout | Deployment starting, scaled to zero, or busy | Check the status in MLIS; wait and retry |
| SSL or certificate error | Internal certificate authority not trusted | Ask your administrator for the CA file and set `SSL_CERT_FILE` and `REQUESTS_CA_BUNDLE` (see Step 5) |
| `ModuleNotFoundError` | Kernel restarted, so `%pip` installs were removed | Re-run Step 2, restart the kernel, continue |
| Answer contains `<think>` text | Reasoning model | Already handled by `strip_reasoning`; keep using it |